# 02 — Exploratory Data Analysis (EDA)

**Purpose:** Understand the data through visualisations and summary statistics before building any models.

---

## Why EDA?

EDA is not optional. It tells you:
- Which features are related to churn
- Whether you have class imbalance
- Which segments have the highest risk
- What feature engineering might help

**Rule:** Never skip EDA. Models built without it often have obvious, fixable problems.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import duckdb

from data_processing import load_clean_data, calculate_churn_rate
from feature_engineering import add_age_group, add_usage_segment
from visualization import (
    plot_churn_distribution, plot_churn_by_plan,
    plot_churn_by_country, plot_churn_by_age_group,
    plot_churn_by_usage, plot_churn_by_payment_method
)

df = load_clean_data()
df = add_age_group(df)
df = add_usage_segment(df)

print(f'Dataset: {len(df):,} customers')
print(f'Churn Rate: {calculate_churn_rate(df):.1f}%')

## 1. Customer Distribution Overview

In [ ]:
print('=== Basic Statistics ===')
df[['age', 'tenure_months', 'usage_frequency', 'monthly_fee',
    'support_tickets', 'payment_failures', 'days_since_login']].describe().round(2)

In [ ]:
# Plan distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['plan'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Customers by Plan')
axes[0].set_xlabel('Plan')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['country'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Customers by Country')
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

df['payment_method'].value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Customers by Payment Method')
axes[2].set_xlabel('Payment Method')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 2. Churn Distribution

In [ ]:
plot_churn_distribution(df).show()

## 3. Churn Rate by Subscription Plan

**Interpretation:** If one plan has significantly higher churn, it may indicate a value-pricing mismatch or poor onboarding for that tier.

In [ ]:
plot_churn_by_plan(df).show()

## 4. Churn Rate by Country

In [ ]:
plot_churn_by_country(df).show()

## 5. Churn Rate by Age Group

In [ ]:
plot_churn_by_age_group(df).show()

## 6. Usage Frequency vs Churn

**Key insight:** Low-engagement customers churn at much higher rates. Usage is one of the strongest early-warning signals.

In [ ]:
plot_churn_by_usage(df).show()

In [ ]:
# Churn rate by usage segment
usage_churn = df.groupby('usage_segment', observed=True)['churned'].mean() * 100
print('Churn rate by usage segment:')
print(usage_churn.round(1))

## 7. Payment Method vs Churn

In [ ]:
plot_churn_by_payment_method(df).show()

## 8. Support Tickets vs Churn

**Interpretation:** More support tickets → higher churn. But note: this is correlation, not causation. Struggling customers file more tickets AND churn more — the root cause may be the same underlying product issue.

In [ ]:
ticket_churn = df.groupby('support_tickets')['churned'].agg(['mean','count']).reset_index()
ticket_churn.columns = ['support_tickets', 'churn_rate', 'customers']
ticket_churn['churn_rate'] = (ticket_churn['churn_rate'] * 100).round(1)

fig = px.bar(ticket_churn, x='support_tickets', y='churn_rate',
             title='Churn Rate by Support Tickets Filed',
             labels={'support_tickets': 'Support Tickets', 'churn_rate': 'Churn Rate (%)'},
             color='churn_rate', color_continuous_scale='RdYlGn_r', text='churn_rate')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 9. Tenure vs Churn

In [ ]:
df['tenure_bucket'] = pd.cut(df['tenure_months'],
    bins=[0, 3, 6, 12, 24, 999],
    labels=['0-3m', '3-6m', '6-12m', '1-2yr', '2yr+'])

tenure_churn = df.groupby('tenure_bucket', observed=True)['churned'].mean() * 100
print('Churn rate by customer tenure:')
print(tenure_churn.round(1))

## 10. SQL Analytics with DuckDB

**What is DuckDB?** An in-process analytical database. You can run SQL directly on your CSV files with zero setup.

In [ ]:
conn = duckdb.connect()

# Register the dataframe so DuckDB can query it
conn.register('customers', df)

# Churn rate by plan — using SQL
result = conn.execute("""
    SELECT
        plan,
        COUNT(*) AS total,
        SUM(churned) AS churned,
        ROUND(AVG(churned) * 100, 2) AS churn_rate_pct,
        ROUND(AVG(monthly_fee), 2) AS avg_fee
    FROM customers
    GROUP BY plan
    ORDER BY churn_rate_pct DESC
""").df()

print('SQL result — Churn by Plan:')
result

In [ ]:
# CTE example — high risk customers
high_risk = conn.execute("""
    WITH risk_scores AS (
        SELECT
            customer_id, plan, usage_frequency, payment_failures, support_tickets,
            (CASE WHEN usage_frequency < 5    THEN 2 ELSE 0 END) +
            (CASE WHEN payment_failures >= 2  THEN 3 ELSE 0 END) +
            (CASE WHEN support_tickets >= 3   THEN 2 ELSE 0 END) AS risk_score
        FROM customers
        WHERE churned = 0
    )
    SELECT *, 
        CASE WHEN risk_score >= 5 THEN 'High'
             WHEN risk_score >= 3 THEN 'Medium'
             ELSE 'Low' END AS risk_level
    FROM risk_scores
    ORDER BY risk_score DESC
    LIMIT 10
""").df()

print('Top 10 at-risk active customers:')
high_risk

## EDA Summary — Key Findings

| Finding | Observed Churn Rate |
|---------|--------------------|
| Low usage (< 5 sessions/month) | Higher than average |
| 2+ payment failures | Significantly higher |
| 4+ support tickets | Elevated |
| Basic plan | Highest among plans |
| New customers (< 3 months) | Variable — honeymoon effect |

**Important:** These are correlations. We will quantify them properly with the ML model in notebook 05.